# SQL reference 6: window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-06-window-functions.ipynb)

The `OVER` clause, filtering on window results without `QUALIFY`, ranking functions, `NTILE` and percentiles, running aggregates and the default frame, `ROWS` and `RANGE` frames including time intervals, `LAG`/`LEAD`, `FIRST_VALUE`/`LAST_VALUE` with `IGNORE NULLS`, and the top-N, deduplication and gaps-and-islands patterns. Practise it in [SQL practice 6: window functions](sql-practice-06-window-functions.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Window basics](#window-basics) | [The `OVER` clause](#the-over-clause), [Filtering on a window result](#filtering-on-a-window-result) |
| [Ranking functions](#ranking-functions) | [`ROW_NUMBER`, `RANK`, `DENSE_RANK`](#row_number-rank-dense_rank), [`NTILE`, `PERCENT_RANK`, `CUME_DIST`](#ntile-percent_rank-cume_dist) |
| [Aggregates over windows](#aggregates-over-windows) | [Running totals and the default frame](#running-totals-and-the-default-frame), [Frames: `ROWS` and `RANGE`](#frames-rows-and-range) |
| [Offset and value functions](#offset-and-value-functions) | [`LAG` and `LEAD`](#lag-and-lead), [`FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`](#first_value-last_value-nth_value) |
| [Patterns](#patterns) | [Top-N per group and deduplication](#top-n-per-group-and-deduplication), [Gaps and islands](#gaps-and-islands) |

## Window basics

### The `OVER` clause

```sql
function(args) OVER ([PARTITION BY keys] [ORDER BY sort_keys] [frame])
function(args) OVER window_name ... WINDOW window_name AS (...)
```

A window function computes a value for **each row** from a set of related rows (its window), without collapsing
rows the way `GROUP BY` does.

| part | meaning |
| --- | --- |
| `PARTITION BY` | split rows into independent groups (omit it for one window over all rows) |
| `ORDER BY` | order rows within each partition; required for ranking and offset functions |
| frame (`ROWS` / `RANGE BETWEEN ...`) | which rows around the current one an aggregate sees |
| `WINDOW w AS (...)` | name a window and reuse it in several functions |

**Notes:** windows are evaluated after `WHERE`, `GROUP BY` and `HAVING`, and before `ORDER BY`/`LIMIT`, so they can
not appear in `WHERE` or `GROUP BY`. Each distinct `PARTITION BY` triggers a shuffle on Spark; reuse one window
definition where possible.

In [ ]:
show("""
SELECT order_id, customer_id, amount,
       sum(amount) OVER (PARTITION BY customer_id) AS customer_total,
       count(*)    OVER ()                         AS all_orders,
       round(100 * amount / sum(amount) OVER (PARTITION BY customer_id), 1) AS pct_of_customer
FROM orders WHERE customer_id IN (1, 2) ORDER BY order_id
""")

### Filtering on a window result

`WHERE` cannot reference a window function. Compute it in a subquery or CTE, then filter outside.

**Notes:** Databricks, Snowflake, BigQuery, Teradata and DuckDB add `QUALIFY`, a `WHERE` for window results
(`... QUALIFY row_number() OVER (...) = 1`). **Spark 4.0 does not support `QUALIFY`**: use the subquery form, which is
portable everywhere.

In [ ]:
show("""
SELECT customer_id, order_id, amount
FROM (SELECT customer_id, order_id, amount,
             row_number() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
      FROM orders)
WHERE rn = 1
ORDER BY customer_id
""")

## Ranking functions

### `ROW_NUMBER`, `RANK`, `DENSE_RANK`

| function | ties | gaps after ties | example for 300, 200, 200, 100 |
| --- | --- | --- | --- |
| `row_number()` | broken arbitrarily (non-deterministic unless the `ORDER BY` is unique) | no | 1, 2, 3, 4 |
| `rank()` | equal rank | yes | 1, 2, 2, 4 |
| `dense_rank()` | equal rank | no | 1, 2, 2, 3 |

**Use:** `row_number` to keep exactly one row per group (deduplication, latest version) with a unique tie-breaker;
`dense_rank` for "the N-th highest **value**"; `rank` for competition-style placings.

In [ ]:
show("""
SELECT name, dept, salary,
       row_number() OVER w AS row_number,
       rank()       OVER w AS rank,
       dense_rank() OVER w AS dense_rank
FROM employees WHERE dept = 'Engineering' OR emp_id = 1
WINDOW w AS (ORDER BY salary DESC)
ORDER BY salary DESC, name
""")

### `NTILE`, `PERCENT_RANK`, `CUME_DIST`

| function | returns |
| --- | --- |
| `ntile(n)` | bucket number 1..n with sizes differing by at most one (quartiles, deciles) |
| `percent_rank()` | `(rank - 1) / (rows - 1)`, from 0 to 1 |
| `cume_dist()` | fraction of rows with a value ≤ the current one, from 1/n to 1 |

In [ ]:
show("""
SELECT order_id, amount, ntile(4) OVER w AS quartile,
       round(percent_rank() OVER w, 2) AS pct_rank, round(cume_dist() OVER w, 2) AS cume_dist
FROM orders WINDOW w AS (ORDER BY amount)
ORDER BY amount
""")

## Aggregates over windows

### Running totals and the default frame

Any aggregate (`sum`, `avg`, `count`, `min`, `max`, `collect_list`, ...) can be a window function. With an `ORDER BY`
and no explicit frame, the frame is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`: a running aggregate in
which **rows with equal sort keys are all included**, so tied rows get the same running total.

| goal | window |
| --- | --- |
| partition total | `sum(x) OVER (PARTITION BY k)` |
| running total, ties together | `sum(x) OVER (PARTITION BY k ORDER BY t)` |
| running total, strictly row by row | `sum(x) OVER (PARTITION BY k ORDER BY t, id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |

**Notes:** `count(DISTINCT x) OVER (...)` is not supported in Spark; use `size(collect_set(x) OVER (...))`.

In [ ]:
show("""
WITH daily AS (SELECT to_date(order_ts) AS day, amount FROM orders)
SELECT day, amount,
       sum(amount) OVER (ORDER BY day) AS range_running,                                       -- same-day rows share a total
       sum(amount) OVER (ORDER BY day, amount ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_running
FROM daily WHERE day <= DATE'2026-01-07' ORDER BY day, amount
""")

### Frames: `ROWS` and `RANGE`

```sql
{ROWS | RANGE} BETWEEN start AND end
start/end: UNBOUNDED PRECEDING | n PRECEDING | CURRENT ROW | n FOLLOWING | UNBOUNDED FOLLOWING
```

| frame | counts | example |
| --- | --- | --- |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | physical rows | 3-row moving average |
| `RANGE BETWEEN 100 PRECEDING AND CURRENT ROW` | rows whose sort **value** is within 100 of the current one | value bands |
| `RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND CURRENT ROW` | a time window over a date or timestamp sort key | trailing 7-day revenue with gaps in the data |

**Notes:** `RANGE` with an offset needs exactly one numeric, date or timestamp `ORDER BY` key. Spark does not support
`GROUPS` frames (Postgres, DuckDB and SQLite do). A trailing time window must use `RANGE`, not `ROWS`, when some days
have no rows or several.

In [ ]:
show("""
SELECT order_id, order_ts, amount,
       round(avg(amount) OVER (ORDER BY order_ts ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS avg_last_3_orders,
       sum(amount) OVER (ORDER BY order_ts RANGE BETWEEN INTERVAL 2 DAYS PRECEDING AND CURRENT ROW) AS last_48h_revenue
FROM orders ORDER BY order_ts
""")

## Offset and value functions

### `LAG` and `LEAD`

`lag(x [, offset [, default]])` returns `x` from the row `offset` rows (default 1) **before** the current row in the
window order; `lead` looks **after**. `default` replaces NULL when there is no such row. Add `IGNORE NULLS` to skip
NULL values.

**Use it for:** period-over-period change, time since the previous event, detecting state changes
(`status <> lag(status)`).

In [ ]:
show("""
WITH daily AS (SELECT to_date(order_ts) AS day, sum(amount) AS revenue FROM orders GROUP BY 1)
SELECT day, revenue,
       lag(revenue) OVER (ORDER BY day) AS prev_revenue,
       revenue - lag(revenue) OVER (ORDER BY day) AS change,
       lead(day) OVER (ORDER BY day) AS next_day
FROM daily ORDER BY day
""")

### `FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`

| function | returns |
| --- | --- |
| `first_value(x) [IGNORE NULLS]` | `x` from the first row of the frame |
| `last_value(x) [IGNORE NULLS]` | `x` from the last row of the frame |
| `nth_value(x, n)` | `x` from the n-th row of the frame |

**Notes:** the default frame ends at the current row, so `last_value` with only an `ORDER BY` returns the **current**
row's value. Use `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` for the true last value, or `first_value`
with the order reversed. `last_value(x) IGNORE NULLS` over a running frame is the standard **forward fill**.

In [ ]:
show("""
SELECT customer_id, order_id, status,
       first_value(status) OVER w_running AS first_status,
       last_value(status)  OVER w_running AS last_value_default_frame,
       last_value(status)  OVER (PARTITION BY customer_id ORDER BY order_ts
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_status
FROM orders WHERE customer_id IN (1, 2)
WINDOW w_running AS (PARTITION BY customer_id ORDER BY order_ts)
ORDER BY customer_id, order_ts
""")
show("""
WITH readings AS (SELECT * FROM VALUES (1, 20.5), (2, NULL), (3, NULL), (4, 21.0), (5, NULL) AS r(minute, temp))
SELECT minute, temp, last_value(temp) IGNORE NULLS OVER (ORDER BY minute) AS filled FROM readings ORDER BY minute
""")

## Patterns

### Top-N per group and deduplication

| task | pattern |
| --- | --- |
| top N rows per group | `row_number() OVER (PARTITION BY g ORDER BY x DESC, id) <= N` in a subquery |
| top N **values** per group (ties included) | `dense_rank() ... <= N` |
| keep the latest version of each key | `row_number() OVER (PARTITION BY key ORDER BY updated_at DESC, seq DESC) = 1` |
| delete exact duplicates | `row_number() OVER (PARTITION BY every_column ORDER BY every_column) = 1`, or `SELECT DISTINCT` |

In [ ]:
show("""
SELECT user_id, event, ts
FROM (SELECT *, row_number() OVER (PARTITION BY user_id, event, ts ORDER BY ts) AS copy FROM events)
WHERE copy = 1 AND user_id = 5
ORDER BY ts
""")

### Gaps and islands

An **island** is a run of consecutive rows that share some property (consecutive days, an unchanged status). Two
classic techniques:

| technique | how |
| --- | --- |
| difference of sequences | for consecutive integers or dates, `value - row_number()` is constant within an island; group by it |
| change flags + running sum | flag rows where a new island starts (`CASE WHEN x <> lag(x) THEN 1 ELSE 0 END`), then `sum(flag) OVER (ORDER BY ...)` numbers the islands |

Sessions, streaks and status periods all reduce to one of these ([time series](sql-reference-08-dates-and-time-series.ipynb)).

In [ ]:
show("""
WITH days AS (SELECT DISTINCT to_date(order_ts) AS day FROM orders),
     grouped AS (SELECT day, date_sub(day, CAST(row_number() OVER (ORDER BY day) AS INT)) AS island FROM days)
SELECT min(day) AS start_day, max(day) AS end_day, count(*) AS days
FROM grouped GROUP BY island ORDER BY start_day
""")

**Next:** practise these entries in [SQL practice 6: window functions](sql-practice-06-window-functions.ipynb).